# Graph Theory for Machine Learning Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Graph class from scratch

In [ ]:
```python

class Graph:

    def __init__(self, n_nodes, directed=False):

        self.n = n_nodes

        self.directed = directed

        self.adj = {i: {} for i in range(n_nodes)}

    def add_edge(self, u, v, weight=1.0):

        self.adj[u][v] = weight

        if not self.directed:

            self.adj[v][u] = weight

    def neighbors(self, node):

        return list(self.adj[node].keys())

    def degree(self, node):

        return len(self.adj[node])

    def adjacency_matrix(self):

        import numpy as np

        A = np.zeros((self.n, self.n))

        for u in range(self.n):

            for v, w in self.adj[u].items():

                A[u][v] = w

        return A

    def degree_matrix(self):

        import numpy as np

        D = np.zeros((self.n, self.n))

        for i in range(self.n):

            D[i][i] = self.degree(i)

        return D

    def laplacian(self):

        return self.degree_matrix() - self.adjacency_matrix()

In [ ]:
```

The adjacency list (`self.adj`) stores neighbors efficiently. The adjacency matrix conversion uses numpy because all the spectral operations need it.

### Step 2: BFS and DFS

In [ ]:
```python

from collections import deque

def bfs(graph, start):

    visited = set()

    order = []

    distances = {}

    queue = deque([(start, 0)])

    visited.add(start)

    while queue:

        node, dist = queue.popleft()

        order.append(node)

        distances[node] = dist

        for neighbor in graph.neighbors(node):

            if neighbor not in visited:

                visited.add(neighbor)

                queue.append((neighbor, dist + 1))

    return order, distances

def dfs(graph, start):

    visited = set()

    order = []

    stack = [start]

    while stack:

        node = stack.pop()

        if node in visited:

            continue

        visited.add(node)

        order.append(node)

        for neighbor in reversed(graph.neighbors(node)):

            if neighbor not in visited:

                stack.append(neighbor)

    return order

In [ ]:
```

BFS uses a deque (double-ended queue) for O(1) popleft. DFS uses a list as a stack. Both visit every node exactly once -- O(V + E) time.

### Step 3: Connected components and Laplacian eigenvalues

In [ ]:
```python

def connected_components(graph):

    visited = set()

    components = []

    for node in range(graph.n):

        if node not in visited:

            order, _ = bfs(graph, node)

            visited.update(order)

            components.append(order)

    return components

def laplacian_eigenvalues(graph):

    import numpy as np

    L = graph.laplacian()

    eigenvalues = np.linalg.eigvalsh(L)

    return eigenvalues

In [ ]:
```

`eigvalsh` is for symmetric matrices -- the Laplacian is always symmetric for undirected graphs. It returns eigenvalues in ascending order. Count the zeros to find the number of connected components.

### Step 4: Spectral clustering

In [ ]:
```python

def spectral_clustering(graph, k=2):

    import numpy as np

    L = graph.laplacian()

    eigenvalues, eigenvectors = np.linalg.eigh(L)

    features = eigenvectors[:, 1:k+1]

    labels = np.zeros(graph.n, dtype=int)

    for i in range(graph.n):

        if features[i, 0] >= 0:

            labels[i] = 0

        else:

            labels[i] = 1

    return labels

In [ ]:
```

For k=2, the sign of the Fiedler vector splits the graph into two clusters. For k>2, you would run k-means on the first k eigenvectors (excluding the trivial all-ones eigenvector).

### Step 5: Message passing

In [ ]:
```python

def message_passing(graph, features, weight_matrix):

    import numpy as np

    A = graph.adjacency_matrix()

    row_sums = A.sum(axis=1, keepdims=True)

    row_sums[row_sums == 0] = 1

    A_norm = A / row_sums

    aggregated = A_norm @ features

    output = aggregated @ weight_matrix

    return output

In [ ]:
```

This is one round of GNN message passing. Each node's new features are the weighted average of its neighbors' features, transformed by the weight matrix. Stack multiple rounds to propagate information further.

## Exercises

In [ ]:
1. **Implement PageRank from scratch.** Start with uniform scores. At each step: score(v) = (1-d)/n + d * sum(score(u)/out_degree(u)) for all u pointing to v. Use d=0.85. Run until convergence (change < 1e-6). Test on a small web graph.

2. **Find communities using spectral clustering.** Create a graph with two clearly separated clusters (e.g., two cliques connected by a single edge). Run spectral clustering and verify it finds the right split. What happens as you add more cross-cluster edges?

3. **Implement Dijkstra's algorithm** for shortest paths in weighted graphs. Compare results to BFS on the same graph with uniform weights.

4. **Build a 2-layer message passing network.** Apply message passing twice with different weight matrices. Show that after 2 rounds, each node has information from its 2-hop neighborhood.

5. **Analyze a real-world graph.** Use the Karate Club graph (34 nodes, 78 edges). Compute degree distribution, Laplacian eigenvalues, and spectral clustering. Compare the spectral clustering result to the known ground truth split.